# QualityPhys - Camera Remote Vital Signs Estimator (CRVSE) Project

## Notebook P3-24: UBFC-Phys Exploration

### What this notebook does
Explores the UBFC-Phys dataset before any preprocessing, to establish its on-disk
layout, file naming, ground-truth formats and metadata. 

### What UBFC-Phys is (from Sabour et al., IEEE TAFFC 2021)
A multimodal social-stress database: 56 subjects, each performing three tasks under a
Trier-Social-Stress-Test protocol — T1 rest, T2 speech, T3 arithmetic — in one of two
scenarios (test = hard, ctrl = easy). Per subject and task it provides:
- an RGB face video (EO-23121C, Motion-JPEG, 35 fps, 1024x1024, ~3 minutes),
- contact BVP from an Empatica E4 wristband (64 Hz),
- contact EDA (4 Hz),
plus a per-subject info file (sex, date/time, scenario) and pre/post self-reported
anxiety scores. 168 videos total (56 x 3). No ECG, no respiration, no biomarkers.

### Why this matters for Phase 3 (the open question)
UBFC-Phys is the dataset whose role is still undecided — train vs held-out. Two facts
from the paper shape both this exploration and that decision:
- The BVP is a WRIST signal and is noisy, especially in T2/T3 where subjects move and
  talk (the authors eliminated 33 subjects from T2 and 28 from T3 on quality criteria).
  So per-recording cardiac_sqi will vary a lot across tasks; T1 is the clean case.
- The three tasks are three distinct conditions (still / speaking / mental arithmetic)
  — the motion + stress diversity that could either strengthen training robustness or
  serve as a hard held-out stress test.



## Imports & Setup

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

UBFC_PHYS_DIR = Path('E:/UBFC_PHYS_dataset')
print('UBFC_PHYS_DIR:', UBFC_PHYS_DIR, '| exists:', UBFC_PHYS_DIR.exists())

# Top-level inventory: subject folders + broad file-type counts.
subject_dirs = sorted([p for p in UBFC_PHYS_DIR.iterdir() if p.is_dir()]) if UBFC_PHYS_DIR.exists() else []
videos = list(UBFC_PHYS_DIR.rglob('*.avi'))
csvs = list(UBFC_PHYS_DIR.rglob('*.csv'))
txts = list(UBFC_PHYS_DIR.rglob('*.txt'))

print('subject dirs:', len(subject_dirs))
print('.avi videos :', len(videos), '| .csv:', len(csvs), '| .txt:', len(txts))

if subject_dirs:
    ex = subject_dirs[0]
    print('\nExample subject folder:', ex.name)
    for f in sorted(ex.iterdir()):
        print('   ', f.name, f'({f.stat().st_size / 1e6:.1f} MB)')
elif videos:
    print('\nNo subject subfolders (flat layout?). First videos:')
    for p in videos[:6]:
        print('   ', p.relative_to(UBFC_PHYS_DIR))

UBFC_PHYS_DIR: E:\UBFC_PHYS_dataset | exists: True
subject dirs: 56
.avi videos : 168 | .csv: 392 | .txt: 56

Example subject folder: s1
    bvp_s1_T1.csv (0.1 MB)
    bvp_s1_T2.csv (0.1 MB)
    bvp_s1_T3.csv (0.1 MB)
    eda_s1_T1.csv (0.0 MB)
    eda_s1_T2.csv (0.0 MB)
    eda_s1_T3.csv (0.0 MB)
    info_s1.txt (0.0 MB)
    selfReportedAnx_s1.csv (0.0 MB)
    vid_s1_T1.avi (4954.7 MB)
    vid_s1_T2.avi (4915.3 MB)
    vid_s1_T3.avi (5001.6 MB)


### File formats and video specs
Confirms the exact content of the ground-truth and metadata files (so the
preprocessing loader reads them correctly) and the real video specs. Checks the BVP
csv structure (plain 64 Hz samples, or an Empatica-E4-style header of start-time +
rate), the EDA csv, the info txt fields, the self-reported-anxiety matrix, and one
video's fps / frame count / resolution via OpenCV properties (no frames decoded or
rendered).

In [2]:
s1 = UBFC_PHYS_DIR / 's1'

def peek(path, n=6):
    with open(path) as f:
        return [f.readline().rstrip('\n') for _ in range(n)]

# --- ground-truth + metadata files ---
for name in ['bvp_s1_T1.csv', 'eda_s1_T1.csv', 'selfReportedAnx_s1.csv']:
    p = s1 / name
    print('===', name, '(first 6 lines) ===')
    for ln in peek(p):
        print('   ', repr(ln))
    try:
        arr = np.loadtxt(p, delimiter=',')
        print('   -> loadtxt shape', arr.shape, '| first vals', np.round(np.ravel(arr)[:5], 3))
    except Exception as e:
        print('   -> loadtxt failed:', e)
    print()

print('=== info_s1.txt ===')
print((s1 / 'info_s1.txt').read_text())
print()

# --- BVP / EDA sample-count sanity vs their stated rates and a ~180 s window ---
bvp = np.loadtxt(s1 / 'bvp_s1_T1.csv', delimiter=',')
eda = np.loadtxt(s1 / 'eda_s1_T1.csv', delimiter=',')
print('bvp_s1_T1 samples:', bvp.size, '| 64Hz*180s =', 64 * 180, '| implied s:', round(bvp.size / 64, 1))
print('eda_s1_T1 samples:', eda.size, '| 4Hz*180s  =', 4 * 180, '| implied s:', round(eda.size / 4, 1))
print()

# --- video specs (properties only; no frames read/rendered) ---
for t in ['T1', 'T2', 'T3']:
    cap = cv2.VideoCapture(str(s1 / f'vid_s1_{t}.avi'))
    fps = cap.get(cv2.CAP_PROP_FPS)
    nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f'vid_s1_{t}.avi: fps {round(fps, 3)} | frames {nfr} | {w}x{h} | implied s {round(nfr / max(fps, 1), 1)}')

=== bvp_s1_T1.csv (first 6 lines) ===
    '90.19'
    '86.56'
    '82.96'
    '79.61'
    '76.75'
    '74.55'
   -> loadtxt shape (11520,) | first vals [90.19 86.56 82.96 79.61 76.75]

=== eda_s1_T1.csv (first 6 lines) ===
    '0.26123'
    '0.26251'
    '0.26251'
    '0.26251'
    '0.26251'
    '0.26251'
   -> loadtxt shape (720,) | first vals [0.261 0.263 0.263 0.263 0.263]

=== selfReportedAnx_s1.csv (first 6 lines) ===
    '2.143,3.286'
    '2.571,3.428'
    '2.778,1.778'
    ''
    ''
    ''
   -> loadtxt shape (3, 2) | first vals [2.143 3.286 2.571 3.428 2.778]

=== info_s1.txt ===
s1
m
test
2019_02_07
11_37_06

bvp_s1_T1 samples: 11520 | 64Hz*180s = 11520 | implied s: 180.0
eda_s1_T1 samples: 720 | 4Hz*180s  = 720 | implied s: 180.0

vid_s1_T1.avi: fps 35.138 | frames 6326 | 1024x1024 | implied s 180.0
vid_s1_T2.avi: fps 35.138 | frames 6325 | 1024x1024 | implied s 180.0
vid_s1_T3.avi: fps 35.138 | frames 6325 | 1024x1024 | implied s 180.0


### BVP quality across tasks (T1 rest / T2 speech / T3 arithmetic)
The paper reports the wrist BVP is noisy in T2/T3 (motion + talking), and cut many
subjects from those tasks. This step quantifies it on the actual signals across all 56
subjects: spectral HR and a cardiac SQI, both taken as the median over fixed 30 s
windows rather than once over the whole 180 s recording.

Two choices here matter. The SQI lobe is floored at 0.15 Hz (~9 bpm), because a bare
3-bin lobe is 3*fs/N wide — over 180 s that spans 1 bpm, so the score measures how
*stationary* the heart rate was rather than how clean the signal is. On these same
signals it moves 0.13 -> 0.80 purely by shortening the window, with no change to the
data, which is enough to invent a quality gap between corpora that differ only in
recording length. And the readout is windowed because a single argmax over three
minutes lets one stretch of motion capture the whole spectrum, which is what parked
several T2/T3 recordings at the 42 bpm band edge.

The per-task SQI distribution informs whether UBFC-Phys enters training (with SQI
soft-weighting, likely T1-heavy) or serves as a held-out stress test. The noisy
threshold below is 0.40 rather than 0.10: it is set where the count of flagged
recordings reproduces the authors' own published exclusions, which is the only external
calibration available for this metric.

In [3]:
def cardiac_metrics(sig, fs, low=0.7, high=3.5, floor_hz=0.15):
    """Spectral HR and a cardiac SQI whose lobe does not shrink with the recording.

    A bare 3-bin fraction has a lobe of 3*fs/N, so on a 180 s recording it spans 1 bpm
    and the score measures how stationary the heart rate was rather than how clean the
    signal is. Across all 56 T1 recordings it moves 0.14 -> 0.70 between a 180 s and a
    10 s window with no change to the data, which made UBFC-Phys look 2.3x worse than
    UBFC-rPPG at native lengths (0.138 against 0.314) when a matched 30 s window puts
    them 0.085 apart. Flooring the lobe at a bandwidth a real heart rate genuinely
    occupies holds the same signals at 0.63-0.70 across every window from 180 s down to
    10 s, and mirrors config.CONFIDENCE_FLOOR_HZ in the app, which exists for this
    reason. The floor stops binding at about 10 s, where the bin spacing reaches
    0.15 Hz on its own.
    """
    x = np.asarray(sig, float); x = x - x.mean()
    if x.std() < 1e-8:
        return np.nan, 0.0
    p = np.abs(np.fft.rfft(x * np.hanning(len(x)))) ** 2
    f = np.fft.rfftfreq(len(x), 1.0 / fs)
    b = (f >= low) & (f <= high)
    if not b.any() or p[b].sum() <= 0:
        return np.nan, 0.0
    band, freqs = p[b], f[b]
    pk = int(np.argmax(band))
    lobe = max(1, int(round(floor_hz / (f[1] - f[0]))))
    return (float(freqs[pk] * 60),
            float(band[max(0, pk - lobe):pk + lobe + 1].sum() / band.sum()))


def windowed_metrics(sig, fs, seconds=30.0):
    """Median SQI and HR over fixed-length windows, for cross-corpus comparison.

    Corpora differ in recording length, so a whole-recording score compares duration as
    much as quality; fixing the window removes that. The median across windows also
    limits how far one stretch of motion can move the reading -- the whole-recording
    argmax parked several T2/T3 recordings on the 42 bpm band edge, and windowing lifts
    the per-task minima to 46 and 49 bpm. It does not rescue them: T2/T3 medians still
    sit below T1 (66.0 and 72.5 against 81.5) because the recordings that read low are
    the noisy ones. Only pairing within subject, on recordings that clear the SQI gate,
    recovers the direction physiology predicts -- see the paired-comparison cell.
    """
    n = int(seconds * fs)
    if n > len(sig):
        return cardiac_metrics(sig, fs)
    rows = [cardiac_metrics(sig[i:i + n], fs)
            for i in range(0, len(sig) - n + 1, n)]
    hr = [h for h, _ in rows if np.isfinite(h)]
    return (float(np.median(hr)) if hr else np.nan,
            float(np.median([s for _, s in rows])))

BVP_FS = 64
rows = []
for sd in subject_dirs:
    for t in ['T1', 'T2', 'T3']:
        bp = sd / f'bvp_{sd.name}_{t}.csv'
        if not bp.exists():
            continue
        bvp = np.loadtxt(bp, delimiter=',')
        hr, sqi = windowed_metrics(bvp, BVP_FS, 30.0)
        rows.append((sd.name, t, len(bvp), round(hr, 1) if np.isfinite(hr) else None, round(sqi, 3)))

df = pd.DataFrame(rows, columns=['subject', 'task', 'n', 'hr_bpm', 'sqi'])
print('recordings measured:', len(df))
print('\nper-task SQI:')
print(df.groupby('task')['sqi'].agg(['mean', 'median', 'min', 'max', 'count']).round(3))
print('\nper-task HR (bpm):')
# Median alongside mean: 8 T2 and 4 T3 recordings still read below 55 bpm, where the
# wrist signal loses the pulse to motion and the spectrum's low-frequency energy wins.
print(df.dropna(subset=['hr_bpm']).groupby('task')['hr_bpm']
        .agg(['mean', 'median', 'min', 'max']).round(1))
# 0.40 on the floored 30 s SQI, not 0.10: the floor lifts every score, and this
# threshold independently reproduces the authors' own quality exclusions -- 33 of 56
# for T2 against their 33, and 27 against their 28 for T3. That is the closest thing
# to an external calibration this metric has, and the unfloored version missed it,
# flagging 46 and 45.
NOISY_SQI = 0.40
print(f'\nSQI < {NOISY_SQI:.2f} (noisy) by task:')
print(df[df['sqi'] < NOISY_SQI].groupby('task').size())
print('(paper eliminated 33/56 from T2 and 28/56 from T3 on quality criteria)')

recordings measured: 168

per-task SQI:
       mean  median    min    max  count
task                                    
T1    0.649   0.699  0.301  0.866     56
T2    0.411   0.384  0.210  0.743     56
T3    0.434   0.408  0.239  0.859     56

per-task HR (bpm):
      mean  median   min    max
task                           
T1    81.7    81.5  58.0  104.0
T2    68.6    66.0  46.0  100.0
T3    74.6    72.5  49.0  106.0

SQI < 0.40 (noisy) by task:
task
T1     3
T2    33
T3    27
dtype: int64
(paper eliminated 33/56 from T2 and 28/56 from T3 on quality criteria)


In [4]:
# Per-task HR compares different subjects in each column, because which recordings are
# clean differs by task -- and the noisy ones read low, which is what makes the stress
# tasks appear to lower heart rate. The honest comparison is within subject, restricted
# to recordings whose reference is trustworthy in both conditions.
NOISY_SQI = 0.40
paired = {s: dict(zip(g.task, zip(g.hr_bpm, g.sqi))) for s, g in df.groupby('subject')}

print(f'within-subject change from T1, both recordings SQI >= {NOISY_SQI:.2f}:')
for task in ('T2', 'T3'):
    deltas = [v[task][0] - v['T1'][0] for v in paired.values()
              if 'T1' in v and task in v
              and v['T1'][1] >= NOISY_SQI and v[task][1] >= NOISY_SQI
              and v['T1'][0] is not None and v[task][0] is not None]
    if deltas:
        print(f'  T1 -> {task}: n={len(deltas):2d}  '
              f'mean {np.mean(deltas):+5.1f}  median {np.median(deltas):+5.1f} bpm')
    else:
        print(f'  T1 -> {task}: no subject clears the gate in both conditions')

print('\nrecordings reading below 55 bpm (motion capturing the spectrum):')
print(df[df['hr_bpm'] < 55].groupby('task').size().to_dict() or 'none')

within-subject change from T1, both recordings SQI >= 0.40:
  T1 -> T2: n=22  mean  -1.8  median  +1.0 bpm
  T1 -> T3: n=28  mean  -0.8  median  +2.0 bpm

recordings reading below 55 bpm (motion capturing the spectrum):
{'T2': 8, 'T3': 4}


### Conclusion: findings and decision

**Layout and formats (confirmed).** UBFC-Phys is one folder per subject (`s{1..56}`),
each holding three task videos (`vid_s{N}_T{1,2,3}.avi`), three BVP csvs (`bvp_...`,
64 Hz), three EDA csvs (`eda_...`, 4 Hz), an `info_s{N}.txt` (subject / gender m|f /
scenario test|ctrl / date / time) and `selfReportedAnx_s{N}.csv` (3x2: cognitive /
somatic / self-confidence, pre and post). Videos are Motion-JPEG, 1024x1024, 35.138 fps,
~6,325 frames = 180.0 s. The BVP is bare 64 Hz values (11,520 = 180.0 s, no
header/timestamp) spanning the same 180 s window as the video, so it aligns to frames by
proportional resampling (as done for UBFC-rPPG) with no clock-offset issue. Full set is
56 x 3 = 168 videos, ~5 GB each (~840 GB raw). No ECG, no respiration, no biomarkers.

**BVP quality.** Median floored SQI over 30 s windows: **T1 rest 0.70, T2 speech 0.38,
T3 arithmetic 0.41** (means 0.65 / 0.41 / 0.43). The rest/stress separation is large and
in the expected direction; the wrist signal degrades under speech and arithmetic exactly
as the paper describes.

The 0.40 threshold is not arbitrary. Counting recordings below it gives **3/56 (T1),
33/56 (T2), 27/56 (T3)** against the authors' own quality-based eliminations of 33/56
from T2 and 28/56 from T3 — an exact match on T2 and within one on T3, from an entirely
independent criterion. The unfloored 3-bin score flags 46 and 45, over-rejecting by 13
and 17, which is the clearest evidence that the floor is the right correction rather
than a convenient one.

Two earlier claims in this notebook do not survive that correction and have been
withdrawn. UBFC-Phys is **not** "the noisiest cardiac reference in the corpus": the
2.3x gap against UBFC-rPPG (0.138 against 0.314) was a comparison of a 180 s recording
against 60 s ones using a metric whose lobe narrows with length. At a matched 30 s
window the two sit 0.085 apart, and the gap runs 0.05 to 0.12 depending on the window
chosen — a real but modest quality difference, not a different class of signal. The DLCN ~0.40 / UBFC-rPPG 0.33 /
MCD 0.17 figures carry the same confound and should not be ranked against each other
until recomputed at one common window.

**Heart rate across tasks is a selection artefact, not a physiological one.** Per-task
median HR reads T1 81.5, T2 66.0, T3 72.5 bpm — stress tasks *below* rest, which is
backwards, with 8 T2 and 4 T3 recordings still reading under 55 bpm where motion energy
dominates the spectrum. But that is a between-subject comparison across columns whose
membership differs by quality. Restricting to subjects whose T1 **and** stress recording
both clear SQI 0.40, and pairing within subject:

| comparison | n | median change |
|---|---|---|
| T1 -> T2 speech | 22 | **+1.0 bpm** |
| T1 -> T3 arithmetic | 28 | **+2.0 bpm** |

The direction flips to the small elevation physiology predicts. Any per-task HR statistic
on this corpus must be paired within subject and restricted to clean recordings, or it
reports which recordings failed rather than what the heart rate did.

**Reference usability against our own gates.** Resampling the reference onto the frame
grid and windowing it at CLIP_LEN/WINDOW_STRIDE, then through the same
`aggregate_windows` the app uses: **T1 54/56 accepted** (median usable fraction 0.865),
T2 33/56, T3 41/56. T1 is a clean quantitative test set rather than merely a qualitative
one, with two references the gates refuse outright. T2/T3 yield a usable subset — a
majority in both cases — whose membership is itself informative, and the refused count
belongs in any error figure quoted from them.

**Decision - role: held-out, not training.** Unchanged, but on different grounds than
the first draft. The reason is not a catastrophically noisy reference — T1 is clean. It
is that the reference is a **wrist** E4 signal rather than a finger or contact sensor,
and that T2/T3 add speech and mental-arithmetic motion that the authors themselves judged
disqualifying for roughly half their subjects. That makes UBFC-Phys a weak supervised
target but a strong motion-and-stress robustness probe. The whole dataset (all three
tasks, all subjects) is preprocessed anyway so the material is on hand: T1 supports a
quantitative cross-dataset resting comparison, while T2/T3 support graceful-degradation
checks, read on the accepted subset and reported with the count that was refused.

One caveat carries into any evaluation: at 35.138 fps a 160-frame clip covers **4.55 s**
here against 5.33 s in every 30 fps corpus, and `decimation_stride` leaves it untouched
because `round(35.138 / 30) = 1`. UBFC-Phys error figures therefore carry a
temporal-extent difference on top of the reference difference, and must be reported with
it rather than compared naively against UBFC-rPPG.
